# Task 5 — Hands-on: First Live Connection

**Ali Raza** · Roll No: **su92-bsdsm-f23-018** · Section: **BSDS-7A**
Data Warehouse & BI — Week 1

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": "#9aa4b2", "axes.labelcolor": "#1f2937",
    "text.color": "#1f2937", "xtick.color": "#4b5563", "ytick.color": "#4b5563",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e5e7eb", "grid.linewidth": 0.7,
    "axes.axisbelow": True, "font.size": 10,
})
INK, BLUE, RED, GREY = "#1f2937", "#2563eb", "#dc2626", "#9ca3af"
os.makedirs("charts", exist_ok=True)
NOTE = "DWBI Week 1 · synthetic course data"


## Before you load — answered before opening Power BI

1. **Which column tells us *what product* was sold?** `Sales.ProductID` — the foreign key; the *name* lives in `Products`.
2. **Where does the region's *name* live?** In `Regions.RegionName` — `Sales` holds only `RegionID`.
3. **The slicer is Product Category — which table, and why not Sales?** From **`Products`**, because `Category` doesn't exist in `Sales` at all. Slicing has to come through the relationship.

## 1. Connect — verify the load

Get Data → Excel workbook → select all three sheets → Load. Then confirm:

In [2]:
S = pd.read_excel("data/DWBI_W1_Regional_Sales.xlsx", sheet_name="Sales")
P = pd.read_excel("data/DWBI_W1_Regional_Sales.xlsx", sheet_name="Products")
R = pd.read_excel("data/DWBI_W1_Regional_Sales.xlsx", sheet_name="Regions")
print(f"Sales: {len(S)} rows (expect 480) | Products: {len(P)} (expect 15) | Regions: {len(R)} (expect 4)")
assert (len(S), len(P), len(R)) == (480, 15, 4)
print("Row counts confirmed:", S.shape, P.shape, R.shape)

Sales: 480 rows (expect 480) | Products: 15 (expect 15) | Regions: 4 (expect 4)
Row counts confirmed: (480, 7) (15, 4) (4, 4)


## 2. Relationships — one side, many side, and how you can tell

| Relationship | One side | Many side | How you can tell |
|---|---|---|---|
| `ProductID` | `Products` | `Sales` | `ProductID` is **unique** in Products (15 distinct = 15 rows: it's the key); it **repeats** in Sales (480 rows, 15 values) |
| `RegionID` | `Regions` | `Sales` | `RegionID` is **unique** in Regions (4 distinct = 4 rows); it **repeats** in Sales |

Rule of thumb: the table where the key is unique is the **one**; the table where it repeats is the **many**. Both detected correctly — no fix needed.

In [3]:
print("Products.ProductID unique:", P.ProductID.is_unique, "| distinct:", P.ProductID.nunique())
print("Sales.ProductID repeats:", not S.ProductID.is_unique, "| distinct:", S.ProductID.nunique())
print("Regions.RegionID unique:", R.RegionID.is_unique, "| distinct:", R.RegionID.nunique())
print("Sales.RegionID repeats:", not S.RegionID.is_unique, "| distinct:", S.RegionID.nunique())
print("Every Sales.ProductID resolves:", S.ProductID.isin(P.ProductID).all())
print("Every Sales.RegionID resolves:", S.RegionID.isin(R.RegionID).all())

Products.ProductID unique: True | distinct: 15
Sales.ProductID repeats: True | distinct: 15
Regions.RegionID unique: True | distinct: 4
Sales.RegionID repeats: True | distinct: 4
Every Sales.ProductID resolves: True
Every Sales.RegionID resolves: True


## 3. Visualise — Total Sales by Region (Region *name*, sorted, finding title)

In [4]:
m = S.merge(R[["RegionID", "RegionName"]], on="RegionID")
tot = m.groupby("RegionName").SalesAmount.sum().sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.barh(tot.index, tot.values, color=BLUE, edgecolor="white", zorder=3)
for i, v in enumerate(tot.values):
    ax.text(v + 12000, i, f"{v:,.0f}", va="center", fontsize=9, color="#4b5563")
ax.set_xlabel("Total sales (Rs)"); ax.set_xlim(0, tot.max()*1.18)
ax.set_title("South outsells every region — nearly 3.4x the West", loc="left")
fig.savefig("charts/t5_sales_by_region.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(tot.sort_values(ascending=False).round(0).to_string())
print("\nIn Power BI: bar chart, Y = RegionName, X = SUM(SalesAmount), sort descending.")

RegionName
South      1118940
Central     943410
North       637340
West        327300

In Power BI: bar chart, Y = RegionName, X = SUM(SalesAmount), sort descending.


![Total sales by region](charts/t5_sales_by_region.png)

## 4. Slice — one category at a time adds back up

In [5]:
cat = S.merge(P[["ProductID", "Category"]], on="ProductID").groupby("Category").SalesAmount.sum().sort_values(ascending=False)
grand = S.SalesAmount.sum()
print(cat.round(0).to_string())
print(f"\nSum of categories: {cat.sum():,.0f}")
print(f"Card with no slicer: {grand:,.0f}")
print("Slicer check PASSED:", abs(cat.sum() - grand) < 0.01, "(selecting every category adds back to the total)")

Category
Electronics    1762350
Grocery         965460
Dairy           157920
Beverages        82340
Snacks           58920

Sum of categories: 3,026,990
Card with no slicer: 3,026,990
Slicer check PASSED: True (selecting every category adds back to the total)


## 5. Validation — all four checks, written down

| Check | Result |
|---|---|
| Card with no slicer = SUM of `SalesAmount` over 480 rows | **3,026,990** — matches the source rows exactly |
| Every category, one at a time, adds back to the total | **3,026,990** — reconciles (see above) |
| The source file is unchanged | slicing filters the *view*; the `.xlsx` on disk is untouched — re-summing it gives the same 3,026,990 |
| No errors or blanks in any visual | 0 nulls in all three tables; every `ProductID`/`RegionID` resolves; no blank appears |

**`UnitPrice` in both tables — which one?** Here they are identical on every row, so the report's numbers don't change either way. The right choice is still principled: use **`SalesAmount`** (already = Quantity × UnitPrice, verified on all 480 rows) for the totals, and treat **`Products.UnitPrice`** as the master for any price *lookup* — the fact table records what was charged; the dimension records the current price. In a real warehouse this duplication is exactly what produces two "correct" answers, so the rule is: one governed source per meaning.

## 6. The interface

**Four panes:** *Fields* (tables/columns/measures to drag in), *Visualizations* (chart types + field wells), *Filters* (page/visual-level filtering), *Canvas* (the report page itself).
**Three views:** *Report* (build what people see), *Data* (inspect the tables), *Model* (check/fix relationships) — used in that order for this task.

## Power BI steps (for the .pbix)

1. Get Data → Excel → select Sales, Products, Regions → Load.
2. Model view: confirm the two relationships (Products 1→* Sales, Regions 1→* Sales).
3. Report view: bar chart (RegionName × SUM(SalesAmount), sorted), card (SUM(SalesAmount)), slicer (Products[Category]).
4. Click each category; watch the card and chart update. Save as `DWBI_W1_su92-bsdsm-f23-018.pbix`, screenshot the report page into the document.

## The question to answer — why did it take a minute?

Because of **Task 2's split**: the question ran on an **OLAP-shaped model** — a star schema (fact + dimensions), pre-related, aggregated in memory — instead of on the live checkout database. No locks were taken on any till, no cashier waited, no one's `INSERT` queued behind my `SUM`. At 6pm on a Friday against the live branch database, the same query would have done what the CEO's report did in Task 2: long shared locks, buffer-pool flood, and every billing counter in the country slowing down behind it.

## Deliverable — `DWBI_W1_su92-bsdsm-f23-018.pbix`

The report file was built with the same model defined above: 3 tables (Sales 480 / Products 15 / Regions 4),
2 relationships (`Products[ProductID] 1→* Sales[ProductID]`, `Regions[RegionID] 1→* Sales[RegionID]`),
measure `Total Sales = SUM(Sales[SalesAmount])`, and a "Regional Sales" page with the bar chart, the card,
and the Category slicer. Validated by reading the file back: all row counts and the grand total
(Rs 3,026,990) match the source workbook.

![Report page](charts/t5_report_page.png)